In [3]:
import numpy as np
import timeit
import torch
import torch.nn.functional as F

from torch import nn

torch.manual_seed(1)
np.random.seed(1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
nt = torch.nested.nested_tensor([torch.arange(12).reshape(2, 6), 
                                 torch.arange(18).reshape(3, 6)], dtype=torch.float, device=device)
print(f"{nt=}")

nt=nested_tensor([
  tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
          [ 6.,  7.,  8.,  9., 10., 11.]]),
  tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
          [ 6.,  7.,  8.,  9., 10., 11.],
          [12., 13., 14., 15., 16., 17.]])
])


In [5]:
padded_out_tensor = torch.nested.to_padded_tensor(nt, padding=0.0)
print(f"{padded_out_tensor=}")

padded_out_tensor=tensor([[[ 0.,  1.,  2.,  3.,  4.,  5.],
         [ 6.,  7.,  8.,  9., 10., 11.],
         [ 0.,  0.,  0.,  0.,  0.,  0.]],

        [[ 0.,  1.,  2.,  3.,  4.,  5.],
         [ 6.,  7.,  8.,  9., 10., 11.],
         [12., 13., 14., 15., 16., 17.]]])


In [6]:
print(f"nt is nested: {nt.is_nested}")
print(f"padded_out_tensor is nested: {padded_out_tensor.is_nested}")

nt is nested: True
padded_out_tensor is nested: False


In [ ]:
# It is common to construct nestedtensors from batches of irregularly shaped 
# tensors. i.e. dimension 0 is assumed to be the batch dimension. 
# Indexing dimension 0 gives back the first underlying tensor component.

print("First underlying tensor component:", nt[0], sep='\n')
print("last column of 2nd underlying tensor component:", nt[1, :, -1], sep='\n')

# When indexing a nestedtensor's 0th dimension, the result is a regular tensor.
print(f"First underlying tensor component is nested: {nt[0].is_nested}")

First underlying tensor component:
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
        [ 6.,  7.,  8.,  9., 10., 11.]])
last column of 2nd underlying tensor component:
tensor([ 5., 11., 17.])
First underlying tensor component is nested: False


In [12]:
# Let's demonstrate that slicing along dimension 0 (i.e., 
# trying to get a combined view
# of the underlying tensor components) is not supported.
try:
  # This attempts to slice the batch dimension of the nested tensor,
  # which is currently not allowed.
  combined_view = nt[0:2]
  print("Combined view:")
  print(combined_view)
except Exception as e:
  print("Error occurred while slicing in dimension 0:", e)

Error occurred while slicing in dimension 0: Could not run 'aten::slice.Tensor' with arguments from the 'NestedTensorCPU' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'aten::slice.Tensor' is only available for these backends: [CPU, CUDA, HIP, XLA, MPS, IPU, XPU, HPU, VE, Lazy, MTIA, PrivateUse1, PrivateUse2, PrivateUse3, Meta, FPGA, MAIA, Vulkan, Metal, QuantizedCPU, QuantizedCUDA, QuantizedHIP, QuantizedXLA, QuantizedMPS, QuantizedIPU, QuantizedXPU, QuantizedHPU, QuantizedVE, QuantizedLazy, QuantizedMTIA, QuantizedPrivateUse1, QuantizedPrivateUse2, QuantizedPrivateUse3, QuantizedMeta, CustomRNGKeyId, MkldnnCPU, SparseCPU, SparseCUDA, SparseHIP, SparseXLA, SparseMPS, SparseIPU, SparseXPU, SparseHPU, SparseVE, SparseLazy, SparseMTIA, SparsePrivateUse1, Spa

In [14]:
nt

nested_tensor([
  tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
          [ 6.,  7.,  8.,  9., 10., 11.]]),
  tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
          [ 6.,  7.,  8.,  9., 10., 11.],
          [12., 13., 14., 15., 16., 17.]])
])

In [ ]:
# The semantics for nestedtensors are similar, except that -1 no longer infers. 
# Instead, it inherits the old size (here 2 for nt[0] and 3 for nt[1]). 
# -1 is the only legal size to specify for a jagged dimension.
nt_reshaped = nt.reshape(2, -1, 2, 3)
print(f"{nt_reshaped=}")

nt_reshaped=nested_tensor([
  tensor([[[ 0.,  1.,  2.],
           [ 3.,  4.,  5.]],
  
          [[ 6.,  7.,  8.],
           [ 9., 10., 11.]]]),
  tensor([[[ 0.,  1.,  2.],
           [ 3.,  4.,  5.]],
  
          [[ 6.,  7.,  8.],
           [ 9., 10., 11.]],
  
          [[12., 13., 14.],
           [15., 16., 17.]]])
])


In [19]:
nt_transposed = nt_reshaped.transpose(1, 2)
print(f"{nt_transposed=}")

nt_transposed=nested_tensor([
  tensor([[[ 0.,  1.,  2.],
           [ 6.,  7.,  8.]],
  
          [[ 3.,  4.,  5.],
           [ 9., 10., 11.]]]),
  tensor([[[ 0.,  1.,  2.],
           [ 6.,  7.,  8.],
           [12., 13., 14.]],
  
          [[ 3.,  4.,  5.],
           [ 9., 10., 11.],
           [15., 16., 17.]]])
])


In [ ]:
# Demonstrate that transposing a nestedtensor involving dimension 0 is not supported.
# Attempting to swap dimension 0 with another dimension should raise an error.
try:
  # This operation is not allowed since dimension 0 is the batch dimension.
  nt_invalid_transpose = nt.transpose(0, 1)
  print("Transposed nestedtensor:")
  print(nt_invalid_transpose)
except Exception as e:
  print("Error occurred while transposing along dimension 0:", e)

Error occurred while transposing along dimension 0: Nested tensor dimension 0 cannot be transposed


In [30]:
print(f"{nt_transposed=}")

nt_transposed=nested_tensor([
  tensor([[[ 0.,  1.,  2.],
           [ 6.,  7.,  8.]],
  
          [[ 3.,  4.,  5.],
           [ 9., 10., 11.]]]),
  tensor([[[ 0.,  1.,  2.],
           [ 6.,  7.,  8.],
           [12., 13., 14.]],
  
          [[ 3.,  4.,  5.],
           [ 9., 10., 11.],
           [15., 16., 17.]]])
])


In [29]:
nt_mm = torch.nested.nested_tensor([torch.arange(24).reshape(2, 3, 4), 
                                    torch.arange(30).reshape(2, 3, 5)], dtype=torch.float, device=device)
print(f"{nt_mm=}")

nt_mm=nested_tensor([
  tensor([[[ 0.,  1.,  2.,  3.],
           [ 4.,  5.,  6.,  7.],
           [ 8.,  9., 10., 11.]],
  
          [[12., 13., 14., 15.],
           [16., 17., 18., 19.],
           [20., 21., 22., 23.]]]),
  tensor([[[ 0.,  1.,  2.,  3.,  4.],
           [ 5.,  6.,  7.,  8.,  9.],
           [10., 11., 12., 13., 14.]],
  
          [[15., 16., 17., 18., 19.],
           [20., 21., 22., 23., 24.],
           [25., 26., 27., 28., 29.]]])
])


In [32]:
nt3 = torch.matmul(nt_transposed, nt_mm)
print(f"Result of Matmul:\n {nt3}")

Result of Matmul:
 nested_tensor([
  tensor([[[ 20.,  23.,  26.,  29.],
           [ 92., 113., 134., 155.]],
  
          [[200., 212., 224., 236.],
           [488., 518., 548., 578.]]]),
  tensor([[[  25.,   28.,   31.,   34.,   37.],
           [ 115.,  136.,  157.,  178.,  199.],
           [ 205.,  244.,  283.,  322.,  361.]],
  
          [[ 250.,  262.,  274.,  286.,  298.],
           [ 610.,  640.,  670.,  700.,  730.],
           [ 970., 1018., 1066., 1114., 1162.]]])
])


In [42]:
nt4 = F.dropout(nt3, 0.1)
print(f"Result of Dropout:\n {nt4}")

Result of Dropout:
 nested_tensor([
  tensor([[[ 22.2222,  25.5556,  28.8889,  32.2222],
           [102.2222, 125.5556, 148.8889, 172.2222]],
  
          [[222.2222, 235.5556, 248.8889,   0.0000],
           [542.2222, 575.5556, 608.8889, 642.2222]]]),
  tensor([[[  27.7778,   31.1111,   34.4444,   37.7778,   41.1111],
           [ 127.7778,  151.1111,  174.4445,  197.7778,  221.1111],
           [ 227.7778,  271.1111,  314.4445,    0.0000,  401.1111]],
  
          [[ 277.7778,  291.1111,  304.4445,  317.7778,  331.1111],
           [ 677.7778,  711.1111,    0.0000,  777.7778,  811.1111],
           [1077.7778,    0.0000, 1184.4445, 1237.7778,    0.0000]]])
])


In [35]:
nt5 = F.softmax(nt4, -1)
print(f"Result of Softmax:\n {nt5}")

Result of Softmax:
 nested_tensor([
  tensor([[[1.2273e-03, 3.4403e-02, 9.6437e-01, 2.7413e-13],
           [3.9754e-31, 5.4066e-21, 7.3530e-11, 1.0000e+00]],
  
          [[4.2483e-18, 2.6231e-12, 1.6196e-06, 1.0000e+00],
           [0.0000e+00, 1.1144e-29, 0.0000e+00, 1.0000e+00]]]),
  tensor([[[1.5637e-06, 4.3834e-05, 1.3503e-18, 3.4444e-02, 9.6551e-01],
           [2.9231e-41, 3.9754e-31, 5.4067e-21, 7.3530e-11, 1.0000e+00],
           [0.0000e+00, 0.0000e+00, 2.2969e-38, 1.5155e-19, 1.0000e+00]],
  
          [[6.8807e-24, 4.2483e-18, 2.6231e-12, 1.6196e-06, 1.0000e+00],
           [0.0000e+00, 3.7835e-44, 1.1144e-29, 3.3383e-15, 1.0000e+00],
           [0.0000e+00, 0.0000e+00, 0.0000e+00, 6.8803e-24, 1.0000e+00]]])
])


In [43]:
sentences = [["goodbye", "padding"],
             ["embrace", "nested", "tensor"]]
vocabulary = {"goodbye": 1.0, "padding": 2.0,
              "embrace": 3.0, "nested": 4.0, "tensor": 5.0}
padded_sentences = torch.tensor([[1.0, 2.0, 0.0],
                                 [3.0, 4.0, 5.0]])
nested_sentences = torch.nested.nested_tensor([torch.tensor([1.0, 2.0]),
                                               torch.tensor([3.0, 4.0, 5.0])])
print(f"{padded_sentences=}")
print(f"{nested_sentences=}")

padded_sentences=tensor([[1., 2., 0.],
        [3., 4., 5.]])
nested_sentences=nested_tensor([
  tensor([1., 2.]),
  tensor([3., 4., 5.])
])


In [44]:
padded_sentences_for_softmax = torch.tensor([[1.0, 2.0, float("-inf")],
                                             [3.0, 4.0, 5.0]])
print(F.softmax(padded_sentences_for_softmax, -1))
print(F.softmax(nested_sentences, -1))

tensor([[0.2689, 0.7311, 0.0000],
        [0.0900, 0.2447, 0.6652]])
nested_tensor([
  tensor([0.2689, 0.7311]),
  tensor([0.0900, 0.2447, 0.6652])
])


In [7]:
import torch

# Demonstrate unflatten on a tensor

# Create a tensor with 24 elements
x = torch.arange(24, dtype=torch.float)
print("Original tensor shape:", x.shape)

# Unflatten the 0-th dimension into a (4, 6) shape tensor
x_unflattened = x.unflatten(0, (4, 6))
print("After unflatten, shape:", x_unflattened.shape)
print(x_unflattened)

# Another example: unflattening a non-batch dimension
# Create a 2D tensor of shape (3, 4)
y = torch.arange(12, dtype=torch.float).reshape(3, 2, 2)
print("\nOriginal tensor y shape:", y.shape)

# Flatten the last dimension to operate on a 1D view
y_flat = y.flatten(start_dim=1)
print("Flattened y shape:", y_flat.shape)

# Now, unflatten the flattened dimension (dim=1) back into (2, 2)
y_unflattened = y_flat.unflatten(1, (2, 2))
print("After unflattening y:")
print("Shape:", y_unflattened.shape)
print(y_unflattened)

Original tensor shape: torch.Size([24])
After unflatten, shape: torch.Size([4, 6])
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
        [ 6.,  7.,  8.,  9., 10., 11.],
        [12., 13., 14., 15., 16., 17.],
        [18., 19., 20., 21., 22., 23.]])

Original tensor y shape: torch.Size([3, 2, 2])
Flattened y shape: torch.Size([3, 4])
After unflattening y:
Shape: torch.Size([3, 2, 2])
tensor([[[ 0.,  1.],
         [ 2.,  3.]],

        [[ 4.,  5.],
         [ 6.,  7.]],

        [[ 8.,  9.],
         [10., 11.]]])
